In [0]:
data = [
    (1,"A",None),
    (2, "B", 5000),
    (3, None, 7000),
    (4, "D", None)
]
df = spark.createDataFrame(data,["id","name","salary"])

In [0]:
df.display()
df.printSchema()

id,name,salary
1,A,null
2,B,5000
3,null,7000
4,D,null


root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)



In [0]:
df.filter(df.salary.isNull()).display()

id,name,salary
1,A,null
4,D,null


In [0]:
df.filter(df.salary.isNull()).select("id").display()

id
1
4


In [0]:
from pyspark.sql.functions import col
df.where(col("salary").isNull()).select("id","name","salary").display()

id,name,salary
1,A,null
4,D,null


In [0]:
df.fillna({"salary": 0}).display()

id,name,salary
1,A,0
2,B,5000
3,null,7000
4,D,0


Removes entire row if any column is NULL

In [0]:
df.dropna().display()

id,name,salary
2,B,5000


In [0]:
df.dropna(subset=["salary"]).display()

id,name,salary
2,B,5000
3,null,7000


### Coalesce is used to fill the values from another column, if the another column value is null then used value in lit
### coalesce(df.col1, df.col2, lit(0))

### 👉 Takes value from:

### col1 → if null → col2 → if null → 0

In [0]:
from pyspark.sql.functions import coalesce,lit
df.withColumn("salary_new",coalesce(df.salary,lit(0))).display()

id,name,salary,salary_new
1,A,null,0
2,B,5000,5000
3,null,7000,7000
4,D,null,0



### 🎯 When to Use?
### Use select() → when you want only specific columns
### Use withColumn() → when you want to add/modify columns + existed coulumns

### 1. Replace NULL salary with average salary

In [0]:
from pyspark.sql.functions import avg

avg_salary = df.select(avg("salary")).collect()[0][0]

df1 = df.fillna({"salary":avg_salary})
df1.display()

id,name,salary
1,A,6000
2,B,5000
3,null,7000
4,D,6000


### 2. Drop rows where name is NULL

In [0]:
df2 = df.dropna(subset=["name"])
df2.display()

id,name,salary
1,A,null
2,B,5000
4,D,null


### 3. Count NULL records

In [0]:
from pyspark.sql.functions import col,count
df.select(count(col("salary")).alias("count")).display()

count
2


### Casting

In [0]:
df = df.withColumn("salary", df.salary.cast("int"))
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)



### Column Operations

In [0]:
from pyspark.sql.functions import col,when
df.select("id","salary").display()

df.withColumn("bouns",col("salary")*0.1).display()
df.withColumn("category",when(col("salary") > 6000,"high").otherwise("Low")).display()

id,salary
1,null
2,5000
3,7000
4,null


id,name,salary,bouns
1,A,null,null
2,B,5000,500.0
3,null,7000,700.0
4,D,null,null


id,name,salary,category
1,A,null,Low
2,B,5000,Low
3,null,7000,high
4,D,null,Low


In [0]:
df = df.withColumn("salary",when(col("salary").isNull(),1000).otherwise(col("salary")))

df = df.withColumn("bouns",col("salary")* 0.1)

df = df.withColumn("category",when(col("salary") > 6000,"High").otherwise("Low"))

# Final output
print("Final Data:")
df.display()

Final Data:


id,name,salary,bouns,category
1,A,1000,100.0,Low
2,B,5000,500.0,Low
3,null,7000,700.0,High
4,D,1000,100.0,Low


In [0]:
def grade(salary):
    if salary is None:
        return "Unknown"
    elif salary > 6000:
        return "A"
    else:
        return "B"

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

grade_udf = udf(grade,StringType())

df = df.withColumn("grade", grade_udf(df.salary))
df.display()

id,name,salary,bouns,category,grade
1,A,1000,100.0,Low,B
2,B,5000,500.0,Low,B
3,null,7000,700.0,High,A
4,D,1000,100.0,Low,B
